# Atelier: videos and pictures on a Colab GPU

This notebook makes the video and picture requests of your Atelier account on Colab's GPU and sends them to your library. Atelier keeps the queue; this notebook only calls your Atelier server, with short requests, so nothing here is exposed to the internet.

**Once:** in Atelier, open **Settings › GPU** and press **Make a token**. Then, in this notebook, open **Secrets** (the key on the left) and add, with *Notebook access* on:
- `ATELIER_URL`: your Atelier's address, such as `https://atelier.example.com`
- `ATELIER_TOKEN`: the token Atelier showed
- `HF_TOKEN` (optional): a Hugging Face token, only for models or LoRAs that ask you to accept their license there
- `CIVITAI_TOKEN` (optional): a Civitai API key (civitai.com, Account settings › API keys), only for LoRAs that ask you to sign in there

**Each time:** choose **Runtime › Change runtime type › A100 GPU** (the first time), then **Runtime › Run all**. The first start installs ComfyUI. Each model downloads the first time a request needs it in the session: about 45 GB for videos (MiniMax H3), 47 GB for pictures (MiniMax H3-Image) and 7 GB more for its prompt expander (Plan). Ask in Atelier: requests start here at once and their progress shows in Atelier's Requests.

The runtime **turns itself off** after the minutes without requests you chose in Atelier, or when you press **Turn off now** there, so it spends no credits waiting. On Colab Pro keep this tab open while you use it (only Pro+ runs with the browser closed).

---
*Português:* em **Configurações › GPU** do Atelier, faça um token. Aqui, em **Secrets** (a chave à esquerda), crie `ATELIER_URL` e `ATELIER_TOKEN` com acesso do notebook ligado (`HF_TOKEN` e `CIVITAI_TOKEN` só para modelos ou LoRAs que pedem conta). Escolha a GPU A100 e use **Ambiente de execução › Executar tudo**. Ele desliga sozinho depois dos minutos sem pedidos que você escolheu no Atelier.

In [ ]:
#@title Start the GPU worker { display-mode: "form" }
import importlib, inspect, sys, urllib.error, urllib.request
from google.colab import userdata

def secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception as error:
        if required:
            raise SystemExit(f'Add the {name} secret (the key on the left) and turn on its notebook access. ({type(error).__name__})')
        return None
    return (value or '').strip() or None

url, token = secret('ATELIER_URL'), secret('ATELIER_TOKEN')
hf_token, civitai_token = secret('HF_TOKEN', False), secret('CIVITAI_TOKEN', False)
base = url.rstrip('/') if url.startswith(('https://', 'http://')) else 'https://' + url.rstrip('/')
endpoint = base if base.endswith('/gpu.php') else base + '/gpu.php'
# The worker comes from your Atelier server, so it always matches its version.
request = urllib.request.Request(endpoint + '?action=worker', data=b'', method='POST',
                                 headers={'Authorization': 'Bearer ' + token, 'User-Agent': 'AtelierGpuWorker'})
try:
    code = urllib.request.urlopen(request, timeout=60).read()
except urllib.error.HTTPError as error:
    raise SystemExit(f'Atelier answered {error.code}: {error.read().decode(errors="replace")[:300]}')
with open('/content/atelier_worker.py', 'wb') as out:
    out.write(code)
sys.path.insert(0, '/content')
import atelier_worker
importlib.reload(atelier_worker)
extra = {'civitai_token': civitai_token} if 'civitai_token' in inspect.signature(atelier_worker.run).parameters else {}
atelier_worker.run(base, token, hf_token=hf_token, **extra)